# Question 7: Hashing and Tree Data Structures

## Introduction

A banking system needs **fast account retrieval**. Hash tables give near O(1) average-case access; Binary Search Trees (BSTs) maintain sorted order but can degrade to O(n); AVL trees fix that degradation by self-balancing. This notebook implements all three and **empirically proves** the AVL-vs-BST performance claim rather than just asserting it in a table.

## (a) Hash Table Implementation

A hash table with **separate chaining** for collision resolution: each bucket holds a list of `(key, value)` pairs.

In [1]:
class HashTable:
    def __init__(self, size=10):
        self.size = size
        self.table = [[] for _ in range(size)]

    def _hash(self, key):
        return sum(ord(c) for c in str(key)) % self.size

    def insert(self, key, value):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                self.table[idx][i] = (key, value)  # update existing key
                return
        self.table[idx].append((key, value))

    def get(self, key):
        idx = self._hash(key)
        for k, v in self.table[idx]:
            if k == key:
                return v
        return None

    def delete(self, key):
        idx = self._hash(key)
        for i, (k, v) in enumerate(self.table[idx]):
            if k == key:
                del self.table[idx][i]
                return True
        return False

ht = HashTable()
ht.insert("ACC001", "Victor")
ht.insert("ACC002", "Alice")
print("ACC001 ->", ht.get("ACC001"))
print("ACC002 ->", ht.get("ACC002"))
print("ACC999 (missing) ->", ht.get("ACC999"))

ACC001 -> Victor
ACC002 -> Alice
ACC999 (missing) -> None


In [2]:
# ---- Tests: insert, update, delete, missing key ----
assert ht.get("ACC001") == "Victor"
ht.insert("ACC001", "Victor Updated")
assert ht.get("ACC001") == "Victor Updated", "insert() must support updating an existing key"
assert ht.delete("ACC002") is True
assert ht.get("ACC002") is None
assert ht.delete("ACC999") is False, "deleting a non-existent key should return False"
print("Hash table insert/update/get/delete: ALL TESTS PASSED ✔")

Hash table insert/update/get/delete: ALL TESTS PASSED ✔


## (b) Collision Handling Techniques

**Chaining** (used above): each bucket stores a *list* of entries, so multiple keys hashing to the same index simply coexist in that bucket's list.

**Linear Probing** (alternative, open-addressing approach): on collision, search sequentially for the next free slot: `index = (index + 1) % table_size`.

Below, chaining is demonstrated explicitly by forcing two keys into the same bucket.

In [3]:
# Force a collision: pick two keys that hash to the same bucket
demo = HashTable(size=5)
k1, k2 = "AB", "BA"   # same character multiset -> same ord() sum -> same bucket
print("hash('AB') =", demo._hash("AB"), " hash('BA') =", demo._hash("BA"))
assert demo._hash(k1) == demo._hash(k2), "expected a deliberate collision for this demo"

demo.insert(k1, "Value for AB")
demo.insert(k2, "Value for BA")
bucket_idx = demo._hash(k1)
print(f"Bucket {bucket_idx} now holds both colliding entries via chaining:", demo.table[bucket_idx])
assert demo.get("AB") == "Value for AB" and demo.get("BA") == "Value for BA"
print("Both keys retrieved correctly despite the collision ✔")

hash('AB') = 1  hash('BA') = 1
Bucket 1 now holds both colliding entries via chaining: [('AB', 'Value for AB'), ('BA', 'Value for BA')]
Both keys retrieved correctly despite the collision ✔


## (c) Binary Search Tree (BST)

A standard, **unbalanced** BST: each insertion places the key according to the binary-search-tree ordering property (left < node < right).

In [4]:
class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None
    def insert(self, key):
        self.root = self._insert(self.root, key)
    def _insert(self, node, key):
        if node is None:
            return Node(key)
        if key < node.key:
            node.left = self._insert(node.left, key)
        elif key > node.key:
            node.right = self._insert(node.right, key)
        return node
    def search(self, key):
        node = self.root
        while node:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False
    def height(self):
        def h(node):
            return 0 if node is None else 1 + max(h(node.left), h(node.right))
        return h(self.root)

tree = BST()
for k in [50, 30, 70, 20, 40, 60, 80]:
    tree.insert(k)

assert tree.search(40) is True
assert tree.search(999) is False
print("BST search tests PASSED ✔  |  height with balanced insert order:", tree.height())

BST search tests PASSED ✔  |  height with balanced insert order: 3


## (d) BST vs AVL Tree Performance — A Real Implementation, Empirically Compared

Rather than just *claiming* "AVL is O(log n), BST is O(n) worst case" in a table, this section **implements a complete AVL tree with rotations** and proves the claim by measuring actual tree heights.

In [5]:
# Worst case for plain BST: inserting already-sorted keys degenerates it into a linked list
bad_bst = BST()
for k in range(1, 11):
    bad_bst.insert(k)
print("BST height after inserting SORTED keys 1..10:", bad_bst.height(), " (linked-list worst case: height == n)")
assert bad_bst.height() == 10

BST height after inserting SORTED keys 1..10: 10  (linked-list worst case: height == n)


In [6]:
class AVLNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        self.height = 1

class AVLTree:
    """Self-balancing BST: after every insertion, rotations restore the
    invariant that left/right subtree heights differ by at most 1."""
    def __init__(self):
        self.root = None

    def _h(self, node):
        return node.height if node else 0

    def _balance_factor(self, node):
        return self._h(node.left) - self._h(node.right) if node else 0

    def _update_height(self, node):
        node.height = 1 + max(self._h(node.left), self._h(node.right))

    def _rotate_right(self, y):
        x, t2 = y.left, y.left.right
        x.right, y.left = y, t2
        self._update_height(y); self._update_height(x)
        return x

    def _rotate_left(self, x):
        y, t2 = x.right, x.right.left
        y.left, x.right = x, t2
        self._update_height(x); self._update_height(y)
        return y

    def insert(self, key):
        self.root = self._insert(self.root, key)

    def _insert(self, node, key):
        if node is None:
            return AVLNode(key)
        if key < node.key:
            node.left = self._insert(node.left, key)
        elif key > node.key:
            node.right = self._insert(node.right, key)
        else:
            return node  # no duplicates

        self._update_height(node)
        balance = self._balance_factor(node)

        if balance > 1 and key < node.left.key:        # Left-Left
            return self._rotate_right(node)
        if balance < -1 and key > node.right.key:       # Right-Right
            return self._rotate_left(node)
        if balance > 1 and key > node.left.key:          # Left-Right
            node.left = self._rotate_left(node.left)
            return self._rotate_right(node)
        if balance < -1 and key < node.right.key:        # Right-Left
            node.right = self._rotate_right(node.right)
            return self._rotate_left(node)
        return node

    def search(self, key):
        node = self.root
        while node:
            if key == node.key:
                return True
            node = node.left if key < node.key else node.right
        return False

    def height(self):
        return self._h(self.root)


good_avl = AVLTree()
for k in range(1, 11):
    good_avl.insert(k)
print("AVL height after inserting the SAME sorted keys 1..10:", good_avl.height(), " (theoretical O(log2 10) ≈ 3.3)")
assert good_avl.height() <= 4, "AVL must stay logarithmically balanced even on sorted input"
assert good_avl.search(7) is True and good_avl.search(999) is False
print("AVL automatically rebalances via rotations — height stays ~log(n) instead of degenerating to n ✔")

AVL height after inserting the SAME sorted keys 1..10: 4  (theoretical O(log2 10) ≈ 3.3)
AVL automatically rebalances via rotations — height stays ~log(n) instead of degenerating to n ✔


In [7]:
# Large-scale empirical comparison: 5,000 random keys
import random
random.seed(42)
keys = random.sample(range(1, 200_000), 5000)

big_bst, big_avl = BST(), AVLTree()
for k in keys:
    big_bst.insert(k)
    big_avl.insert(k)

import math
print(f"n = {len(keys)} random keys")
print(f"  BST height : {big_bst.height()}")
print(f"  AVL height : {big_avl.height()}   (theoretical minimum ~log2(n) = {math.log2(len(keys)):.1f})")
assert big_avl.height() < big_bst.height()
print("\nCONFIRMED: AVL height stays close to the theoretical O(log n) bound; plain BST height grows larger ✔")

n = 5000 random keys
  BST height : 28
  AVL height : 15   (theoretical minimum ~log2(n) = 12.3)

CONFIRMED: AVL height stays close to the theoretical O(log n) bound; plain BST height grows larger ✔


### Summary Table (now backed by the measurements above, not just assumed)

| Feature | BST | AVL Tree |
|---|---|---|
| Balance | Not guaranteed | Self-balancing via rotations |
| Search (worst case) | O(n) — degenerates on sorted input | O(log n) — proven above |
| Insertion (worst case) | O(n) | O(log n) |
| Measured height, sorted 1–10 | 10 | ≤ 4 |
| Measured height, 5,000 random keys | (printed above, larger) | (printed above, ~log₂5000 ≈ 12–17) |

**Conclusion:** AVL trees are strictly preferable for a banking system where account lookups must stay fast (O(log n)) regardless of the order accounts were inserted — a plain BST risks silently degrading to O(n) linear-list behaviour if account numbers happen to be inserted in (or near) sorted order, which is a realistic operational risk (e.g. sequentially-issued account numbers).